# Model training

In [ ]:
# import the necessary libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout

from sentiment_analysis.utils import PATH_CLEANED_TRAINING_DATASET

In [ ]:
# load the cleaned dataset
movie_reviews = pd.read_csv(PATH_CLEANED_TRAINING_DATASET)

In [ ]:
# split into training set (80%), dev set (10%) and test set (10%)
train_df, rem_df = train_test_split(movie_reviews, train_size=0.8, random_state=42)
dev_df, test_df = train_test_split(rem_df, test_size=0.5, random_state=42)

# delete temporary dataframe
del rem_df

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")

In [ ]:
# separate features and targets
X_train = train_df["review"].values
y_train = train_df["sentiment"].values
X_dev = dev_df["review"].values
y_dev = dev_df["sentiment"].values
X_test = test_df["review"].values
y_test = test_df["sentiment"].values

## Tokenize

In [ ]:
word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(X_train)

In [ ]:
# Adding 1 to store dimensions for words for which no pretrained word embeddings exist
vocab_length = len(word_tokenizer.word_index) + 1

In [ ]:
# Convert text to sequences
X_train = word_tokenizer.texts_to_sequences(X_train)
X_dev = word_tokenizer.texts_to_sequences(X_dev)
X_test = word_tokenizer.texts_to_sequences(X_test)

In [ ]:
# Padding all reviews to fixed length 250
maxlen = 250
padding_type = 'post'

X_train = pad_sequences(X_train, padding=padding_type, maxlen=maxlen)
X_dev = pad_sequences(X_dev, padding=padding_type, maxlen=maxlen)
X_test = pad_sequences(X_test, padding=padding_type, maxlen=maxlen)

In [ ]:
print("Sample Padded Sequence (First 10 tokens):")
print(X_train[0][:10])

In [ ]:
# Load the pre-trained GloVe word embeddings and create an Embeddings Dictionary
from numpy import asarray
from numpy import zeros
from sentiment_analysis.utils import PATH_GLOVE_EMBEDDINGS

embeddings_dictionary = dict()
glove_file = open(PATH_GLOVE_EMBEDDINGS, encoding="utf8")

for line in glove_file:
    records = line.split()
    word = records[0]
    vector_dimensions = asarray(records[1:], dtype='float32')
    embeddings_dictionary [word] = vector_dimensions
glove_file.close()

In [ ]:
# Create Embedding Matrix having 100 columns 
# Containing 100-dimensional GloVe word embeddings for all words in our corpus.

embedding_matrix = zeros((vocab_length, 100))
for word, index in word_tokenizer.word_index.items():
    embedding_vector = embeddings_dictionary.get(word)
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

In [ ]:
embedding_matrix.shape

In [ ]:
# Define the LSTM model
model = Sequential([
    Embedding(vocab_length, 100, weights=[embedding_matrix], trainable=False),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

In [ ]:
# Compiling the model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Model Training
batch_size=64
epochs=6
lstm_model_history = model.fit(
    X_train, y_train, 
    batch_size=batch_size, 
    epochs=epochs, 
    verbose=1, 
    validation_data=(X_dev, y_dev),
)

In [ ]:
# Evaluate the model on the dev set
score = model.evaluate(X_dev, y_dev, verbose=1)

In [ ]:
# Model Performance

print("Validation Score:", score[0])
print("Validation Accuracy:", score[1])

In [ ]:
# Model Performance Charts

import matplotlib.pyplot as plt

plt.plot(lstm_model_history.history['accuracy'])
plt.plot(lstm_model_history.history['val_accuracy'])

plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train','dev'], loc='upper left')
plt.show()

plt.plot(lstm_model_history.history['loss'])
plt.plot(lstm_model_history.history['val_loss'])

plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train','dev'], loc='upper left')
plt.show()

In [ ]:
# Saving the model file for use later
from sentiment_analysis.utils import PATH_MODEL

model.save(PATH_MODEL)

# Final evaluation using test datasets

In [ ]:
test_score = model.evaluate(X_test, y_test)

In [ ]:
# Model Performance

print("Validation Score:", score[0])
print("Validation Accuracy:", score[1])

print("Test Score:", test_score[0])
print("Test Accuracy:", test_score[1])